# LR

In [1]:
!pip install scikit-learn scipy joblib

Looking in indexes: http://mirrors.aliyun.com/pypi/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 1.8 MB/s eta 0:00:0000:0100:010m
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.7/37.7 MB 1.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.1/309.1 kB 69.6 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import os

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

In [3]:
DATA_PATH = "FeatureB_Repeated"
OUTPUT_PATH = "LR_FeatureB_Results"

os.makedirs(OUTPUT_PATH, exist_ok=True)

N_REPEATS = 10

C_VALUES = [0.1, 1, 10]

N_INNER_REPEATS = 3
VALID_RATIO = 0.2
BASE_SEED = 42

In [4]:
def get_feature_cols(df):
    return [
        col for col in df.columns
        if col not in ["userId", "movieId", "label"]
    ]

In [5]:
def stratified_split_from_scratch(df, label_col, test_ratio=0.2, random_seed=42):
    rng = np.random.default_rng(random_seed)
    train_indices = []
    test_indices = []
    for label_value in df[label_col].unique():
        label_indices = df[df[label_col] == label_value].index.to_numpy()
        rng.shuffle(label_indices)

        test_size = int(len(label_indices) * test_ratio)

        test_indices.extend(label_indices[:test_size])
        train_indices.extend(label_indices[test_size:])

    train_df = df.loc[train_indices].sample(
        frac=1,
        random_state=random_seed
    ).reset_index(drop=True)

    test_df = df.loc[test_indices].sample(
        frac=1,
        random_state=random_seed
    ).reset_index(drop=True)

    return train_df, test_df

In [6]:
def compute_basic_metrics_from_scratch(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))

    accuracy = (tp + tn) / len(y_true)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0

    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn
    }

In [7]:
def compute_auc_from_scratch(y_true, y_prob):
    y_true = np.array(y_true)
    y_prob = np.array(y_prob)

    sorted_indices = np.argsort(-y_prob)
    y_true_sorted = y_true[sorted_indices]

    pos_count = np.sum(y_true == 1)
    neg_count = np.sum(y_true == 0)

    if pos_count == 0 or neg_count == 0:
        return 0

    tp = 0
    fp = 0

    tpr_list = [0]
    fpr_list = [0]

    for label in y_true_sorted:
        if label == 1:
            tp += 1
        else:
            fp += 1

        tpr_list.append(tp / pos_count)
        fpr_list.append(fp / neg_count)

    auc = 0

    for i in range(1, len(tpr_list)):
        auc += (
            (fpr_list[i] - fpr_list[i - 1])
            *
            (tpr_list[i] + tpr_list[i - 1])
            / 2
        )

    return auc

In [8]:
def train_and_evaluate_lr(train_df, test_df, C_value):
    feature_cols = get_feature_cols(train_df)

    X_train = train_df[feature_cols]
    y_train = train_df["label"]

    X_test = test_df[feature_cols]
    y_test = test_df["label"]

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = LogisticRegression(
        C=C_value,
        penalty="l2",
        solver="liblinear",
        max_iter=1000,
        random_state=42
    )

    model.fit(X_train_scaled, y_train)

    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]

    metrics = compute_basic_metrics_from_scratch(y_test, y_pred)
    metrics["auc"] = compute_auc_from_scratch(y_test, y_prob)

    return metrics

In [9]:
def tune_lr_C_from_scratch(train_df, C_values, n_inner_repeats=3, valid_ratio=0.2, base_seed=100):
    tuning_records = []

    for C_value in C_values:
        inner_f1_scores = []

        for inner_id in range(n_inner_repeats):
            inner_train_df, valid_df = stratified_split_from_scratch(
                train_df,
                label_col="label",
                test_ratio=valid_ratio,
                random_seed=base_seed + inner_id
            )

            metrics = train_and_evaluate_lr(
                train_df=inner_train_df,
                test_df=valid_df,
                C_value=C_value
            )

            inner_f1_scores.append(metrics["f1"])

        tuning_records.append({
            "C": C_value,
            "mean_validation_f1": np.mean(inner_f1_scores),
            "std_validation_f1": np.std(inner_f1_scores, ddof=1)
        })

    tuning_df = pd.DataFrame(tuning_records)

    best_C = tuning_df.sort_values(
        by="mean_validation_f1",
        ascending=False
    ).iloc[0]["C"]

    return best_C, tuning_df

In [10]:
all_results = []
all_tuning_results = []

for repeat_id in range(1, N_REPEATS + 1):

    print("=" * 60)
    print(f"Outer Repeat {repeat_id:02d}")
    print("=" * 60)

    repeat_folder = os.path.join(
        DATA_PATH,
        f"repeat_{repeat_id:02d}"
    )

    train_path = os.path.join(repeat_folder, "feature_B_train.csv")
    test_path = os.path.join(repeat_folder, "feature_B_test.csv")

    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    print("Train shape:", train_df.shape)
    print("Test shape:", test_df.shape)

    best_C, tuning_df = tune_lr_C_from_scratch(
        train_df=train_df,
        C_values=C_VALUES,
        n_inner_repeats=N_INNER_REPEATS,
        valid_ratio=VALID_RATIO,
        base_seed=1000 + repeat_id * 10
    )

    print("Best C:", best_C)

    tuning_df["outer_repeat"] = repeat_id
    all_tuning_results.append(tuning_df)

    final_metrics = train_and_evaluate_lr(
        train_df=train_df,
        test_df=test_df,
        C_value=best_C
    )

    result_row = {
        "outer_repeat": repeat_id,
        "best_C": best_C,
        **final_metrics
    }

    all_results.append(result_row)

    print("Accuracy :", round(final_metrics["accuracy"], 4))
    print("Precision:", round(final_metrics["precision"], 4))
    print("Recall   :", round(final_metrics["recall"], 4))
    print("F1       :", round(final_metrics["f1"], 4))
    print("AUC      :", round(final_metrics["auc"], 4))

Outer Repeat 01
Train shape: (160000, 122)
Test shape: (39999, 122)
Best C: 10.0
Accuracy : 0.6227
Precision: 0.6331
Recall   : 0.5827
F1       : 0.6069
AUC      : 0.6474
Outer Repeat 02
Train shape: (160000, 122)
Test shape: (39999, 122)
Best C: 10.0
Accuracy : 0.6167
Precision: 0.6255
Recall   : 0.5809
F1       : 0.6024
AUC      : 0.6428
Outer Repeat 03
Train shape: (160000, 122)
Test shape: (39999, 122)
Best C: 1.0
Accuracy : 0.6246
Precision: 0.6321
Recall   : 0.5955
F1       : 0.6133
AUC      : 0.6477
Outer Repeat 04
Train shape: (160000, 122)
Test shape: (39999, 122)
Best C: 10.0
Accuracy : 0.6175
Precision: 0.6282
Recall   : 0.5751
F1       : 0.6005
AUC      : 0.6453
Outer Repeat 05
Train shape: (160000, 122)
Test shape: (39999, 122)
Best C: 10.0
Accuracy : 0.6232
Precision: 0.6303
Recall   : 0.5949
F1       : 0.6121
AUC      : 0.6462
Outer Repeat 06
Train shape: (160000, 122)
Test shape: (39999, 122)
Best C: 1.0
Accuracy : 0.6282
Precision: 0.6367
Recall   : 0.5963
F1       : 0

In [11]:
results_df = pd.DataFrame(all_results)
tuning_results_df = pd.concat(all_tuning_results, ignore_index=True)

results_path = os.path.join(OUTPUT_PATH, "LR_FeatureB_repeated_results.csv")
tuning_path = os.path.join(OUTPUT_PATH, "LR_FeatureB_tuning_results.csv")

results_df.to_csv(results_path, index=False, encoding="utf-8-sig")
tuning_results_df.to_csv(tuning_path, index=False, encoding="utf-8-sig")

print("Saved repeated test results to:")
print(results_path)

print("Saved tuning results to:")
print(tuning_path)

results_df

Saved repeated test results to:
LR_FeatureB_Results/LR_FeatureB_repeated_results.csv
Saved tuning results to:
LR_FeatureB_Results/LR_FeatureB_tuning_results.csv


,outer_repeat,best_C,accuracy,precision,recall,f1,tp,tn,fp,fn,auc
0,1,10.0,0.622691,0.633112,0.582691,0.606856,11648,13259,6750,8342,0.647351
1,2,10.0,0.616740,0.625498,0.580940,0.602397,11613,13056,6953,8377,0.642848
2,3,1.0,0.624641,0.632116,0.595498,0.613261,11904,13081,6928,8086,0.647748
3,4,10.0,0.617540,0.628197,0.575088,0.600470,11496,13205,6804,8494,0.645278
4,5,10.0,0.623166,0.630307,0.594897,0.612090,11892,13034,6975,8098,0.646236
5,6,1.0,0.628166,0.636650,0.596298,0.615814,11920,13206,6803,8070,0.651020
6,7,10.0,0.620891,0.629829,0.585593,0.606906,11706,13129,6880,8284,0.647875
7,8,10.0,0.625441,0.630037,0.606903,0.618254,12132,12885,7124,7858,0.649035
8,9,1.0,0.623091,0.633330,0.583842,0.607580,11671,13252,6757,8319,0.647947
9,10,0.1,0.620841,0.630718,0.582191,0.605484,11638,13195,6814,8352,0.648768


In [12]:
summary_records = []

for metric in ["accuracy", "precision", "recall", "f1", "auc"]:
    values = results_df[metric].values

    summary_records.append({
        "metric": metric,
        "mean": np.mean(values),
        "std": np.std(values, ddof=1),
        "standard_error": np.std(values, ddof=1) / np.sqrt(len(values))
    })

summary_df = pd.DataFrame(summary_records)

summary_path = os.path.join(OUTPUT_PATH, "LR_FeatureB_summary.csv")
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

summary_df

,metric,mean,std,standard_error
0,accuracy,0.622321,0.003479,0.001100
1,precision,0.630979,0.003056,0.000966
2,recall,0.588394,0.009603,0.003037
3,f1,0.608911,0.005768,0.001824
4,auc,0.647411,0.002233,0.000706


In [13]:
best_C_frequency = results_df["best_C"].value_counts().reset_index()
best_C_frequency.columns = ["C", "frequency"]

best_C_frequency_path = os.path.join(OUTPUT_PATH, "LR_FeatureB_best_C_frequency.csv")
best_C_frequency.to_csv(best_C_frequency_path, index=False, encoding="utf-8-sig")

best_C_frequency

,C,frequency
0,10.0,6
1,1.0,3
2,0.1,1
